# Running this experiment

See README.md for setup, required inputs, and limitations. Run cells in order from a fresh kernel. Saved outputs have been cleared; historical metrics are not newly verified results. Environment installation and external training commands are manual setup steps.

In [ ]:
import os

os.environ["TF_USE_LEGACY_KERAS"] = "1"  # Required by TensorFlow Hub KerasLayer.

# Override these paths for your local datasets. PROJECT_DATA_DIR changes their base.
INPUT_PATHS = {}

In [ ]:
# Check for GPU
pass  # Environment setup moved to README; use local inputs.

# Import Data and Helper Functions

In [ ]:
# Download Daniel Bourke's helper functions
pass  # Environment setup moved to README; use local inputs.

In [ ]:
# Import series of helper functions for the notebook
from helper_functions import (
    unzip_data,
    create_tensorboard_callback,
)

In [ ]:
# Download the Natural Language Processing with Disaster Tweets dataset from Kaggle (here we use the one from ztm_tf_course which is the same as the one from kaggle)
pass  # Environment setup moved to README; use local inputs.

In [ ]:
# Unzip the data
unzip_data("nlp_getting_started.zip")

In [ ]:
# Turn the csv files into pandas dataframe
import pandas as pd

train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
train_df.head(10)

# Visualize the Data

In [ ]:
# Shuffle the training dataframe
train_df_shuffled = train_df.sample(
    frac=1, random_state=42
)  # frac is the fraction of dataframe to return and random_state value ensures reproducibility

In [ ]:
train_df_shuffled.head(10)

### We are going to write code to find the value of the target column based on the text column

### Inputs (text coumn) --> Machine Learning Algorithm --> Outputs (target column)

In [ ]:
# The test data doesn't have a target column
test_df.head()

In [ ]:
# Check how many examples of each target value we have
train_df.target.value_counts()

### Since we have 2 target values, we are gonna be doing binary classification

### Here, 1 is a real disaster Tweet and 0 is not a real disaster Tweet

In [ ]:
# Total number of samples we have
print(f"Total training samples: {len(train_df)}")
print(f"Total test samples: {len(test_df)}")
print(f"Total samples: {len(train_df) + len(test_df)}")

In [ ]:
# Visualize random training examples
import random  # random generates pseudo-random numbers

random_index = random.randint(
    0, len(train_df) - 5
)  # create random indexes not higher then the number of samples
for row in train_df_shuffled[["text", "target"]][
    random_index : random_index + 5
].itertuples():  # returns the values of row as a tuple and loops over it
    _, text, target = row  # only get these two column values from each row
    print(f"Target: {target}", "(real disaster)" if target > 0 else "(not real disaster)")
    print(f"Text:\n{text}\n")
    print("---\n")

# Split the data into trainig and validation sets

In [ ]:
from sklearn.model_selection import train_test_split

# Use train_test_split to split training data into training and validation sets
train_sentences, val_sentences, train_labels, val_labels = train_test_split(
    train_df_shuffled["text"].to_numpy(),  # converts the dataframe to numpy array
    train_df_shuffled["target"].to_numpy(),
    test_size=0.1,  # dedicates 10% of the samples to validation set
    random_state=42,
)  # random state for reproducibility

In [ ]:
# Check the lengths of the sets
len(train_sentences), len(val_sentences), len(train_labels), len(val_labels)

In [ ]:
# View first 10 training sentences and their labels
train_sentences[:10], train_labels[:10]

# Converting Text Data to Numbers (Tokenization and Embeddings)


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

# Initializing the text vectorizer
text_vectorizer = TextVectorization(
    max_tokens=10000,  # how many words in the vocabulary (all of the different words in your text)
    standardize="lower_and_strip_punctuation",  # how to process text
    split="whitespace",  # how to split the tokens
    ngrams=None,  # create groups of n words
    output_mode="int",  # how to map tokens to numbers
    output_sequence_length=None,  # how long should the output sequence of tokens be
    pad_to_max_tokens=True,
)

### We set the max_tokens to 10,000 and we find the output_sequence_length which will be the average number of tokens per tweet in the training dataset

In [ ]:
# Find the average number of tokens (words) in the training Tweets
round(sum([len(i.split()) for i in train_sentences]) / len(train_sentences))

In [ ]:
# Setup text vectorization variables
max_vocab_Length = 10000  # max number of words in our dataset
max_length = 15  # max length of our sequences will be (how many words per tweet does our model see)
text_vectorizer = TextVectorization(
    max_tokens=max_vocab_Length, output_mode="int", output_sequence_length=max_length
)

In [ ]:
# Fit the text vectorizer to the trainig text
text_vectorizer.adapt(train_sentences)  # adapt() maps text_vectorizer to out data

In [ ]:
# Create sample sentence and tokenize it
sample_sentence = "There's a flood in my street!"
text_vectorizer([sample_sentence])

### What we have done is word level tokenization. Notice the 0's at the end of the returned tensor, this is because we set output_sequence_length=15, meaning no matter the size of the sequence we pass to text_vectorizer, it always returns a sequence with a length of 15.

In [ ]:
# Chose a random sentence from the training dataset and tokenize it
random_sentence = random.choice(train_sentences)
print(
    f"Original Text:\n{random_sentence}\
      \n\nVectorized version:"
)
text_vectorizer([random_sentence])

In [ ]:
# Get the unique words in the vocabulary
words_in_vocab = text_vectorizer.get_vocabulary()  # gets all the unique words in the vocabulary
top_5_words = words_in_vocab[:5]  # most common tokens (UNK token means "unknown" words)
bottom_5_words = words_in_vocab[-5:]  # least common tokens
print(f"Number of words in vocab: {len(words_in_vocab)}")
print(f"Top 5 most common words: {top_5_words}")
print(f"Bottom 5 least common words: {bottom_5_words}")

## Create an Embedding using Embedding Layer

In [ ]:
from tensorflow.keras import layers

# Initialize the Embedding layer
embedding = layers.Embedding(
    input_dim=max_vocab_Length,  # set the input shape
    output_dim=128,  # set size of embedding vector
    embeddings_initializer="uniform",  # default, initialize randomly
    input_length=max_length,
)

In [ ]:
# Get a random sentence from training set
random_sentence = random.choice(train_sentences)
print(
    f"Original Text:\n {random_sentence}\
\n\nEmbedded Version:"
)

# Embed the random sentence (turn it into numerical representaion)
sample_embed = embedding(text_vectorizer([random_sentence]))
sample_embed

### Each token in the sentence gets turned into a length 128 feature vector. Each sentence has 15 tokens. That's why the shape is 1, 15, 128

In [ ]:
# Check out a single token's embedding
sample_embed[0][0]

# Create Models

### Steps to go through:
* Construct the model
* Train the model
* Make predictions with the model
* Track prediction evaluation metrics for later comparison

## Model 0: Naive Bayes

### To create out baseline model to get a benchmark for future experiments to build upon, er create a scikit-learn pipeline using the TF-IDF (term frequency-inverse document frequency) formula to convert our words to numbers and then model them with the Multinominal Naive Bayes algorithm. 

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

# Create tokenization and modelling pipeline
model_0 = Pipeline(
    [
        ("tfidf", TfidfVectorizer()),  # convert words to numbers using tfidf
        ("clf", MultinomialNB()),  # model the text
    ]
)

# Fit the pipeline to training data
model_0.fit(train_sentences, train_labels)

### Evaluate the model to find the baseline metrics

In [ ]:
baseline_score = model_0.score(val_sentences, val_labels)
print(f"Our baseline model achieves an accuracy of: {baseline_score * 100:.2f}%")

In [ ]:
# Make predictions
baseline_preds = model_0.predict(val_sentences)
baseline_preds[:20]

## Creating an evaluation function for our model experiments

Takes an array of predictions and ground truth labels and computes the following:
* Accuracy
* Precision
* Recall
* F1-score

In [ ]:
# Function to evaluate: accuracy, precision, recall, f1-score

from experiment_nlp_with_disaster_tweets import calculate_results

In [ ]:
# Get baseline results
baseline_results = calculate_results(y_true=val_labels, y_pred=baseline_preds)

baseline_results

## Model 1: Simple Dense Model

In [ ]:
# Create tensorboard callback (need to create a new one for each model)
from helper_functions import create_tensorboard_callback

# Create directory to save tensorboard logs
SAVE_DIR = "model_logs"

In [ ]:
# Build the model with the functional API

from tensorflow.keras import layers

inputs = layers.Input(shape=(1,), dtype="string")  # inputs are 1 dimensional strings
x = text_vectorizer(inputs)  # turns input texts into numbers
x = embedding(x)  # create an embedding of the numerized numbers
x = layers.GlobalAveragePooling1D()(x)  # lower the dimensionality of the embeddings
outputs = layers.Dense(
    1, activation="sigmoid"
)(
    x
)  # create output layer, we want binary outputs so we use sigmoid activation (for categorical use softmax)
model_1 = tf.keras.Model(inputs, outputs, name="model_1_dense")  # constructs the model

In [ ]:
# Compile the model
model_1.compile(
    loss="binary_crossentropy",
    optimizer=tf.keras.optimizers.Adam(),
    metrics=["accuracy"],
)

In [ ]:
# Get summary of model_1
model_1.summary()

### We created an embedding of soze 128 for a vocabulary of 10,000. So 128 x 10,000 = 1280000 parameters

In [ ]:
# Fit the model to training data for 5 epochs
model_1.history = model_1.fit(
    train_sentences,
    train_labels,
    epochs=5,
    validation_data=(val_sentences, val_labels),
    callbacks=[
        create_tensorboard_callback(dir_name=SAVE_DIR, experiment_name="simple_dense_model")
    ],
)

In [ ]:
# Check the results
model_1.evaluate(val_sentences, val_labels)

In [ ]:
# Make predictions (these comeback in the form of probabilities)
model_1_pred_probs = model_1.predict(val_sentences)
model_1_pred_probs[:10]

### Since we used a sigmoid function, our predictions come back in the form of probabilities. To convert predictions to classes, we round them with a threshold value of 0.5

In [ ]:
# Turn prediction probabilities into single dimensional tensor of floats
model_1_preds = tf.squeeze(tf.round(model_1_pred_probs))  # squeeze removes single dimensions
model_1_preds[:10]

In [ ]:
# Calculate model_1 metrics
model_1_results = calculate_results(y_true=val_labels, y_pred=model_1_preds)

In [ ]:
model_1_results

In [ ]:
# Is our simple Keras model better than our baseline model
import numpy as np

np.array(list(model_1_results.values())) > np.array(list(baseline_results.values()))

## We creat a new function to compare any model to our baseline model

In [ ]:
# Create a helper function to compare our baseline results to our new models
from experiment_nlp_with_disaster_tweets import compare_baseline_to_new_results

compare_baseline_to_new_results(
    baseline_results=baseline_results, new_model_results=model_1_results
)

### Our first model (model_1) contained an embedding layer which learned a way of representing words as feature vectors by passing over the training data

In [ ]:
# Get the vocabulary from the text vectorization layer
words_in_vocab = text_vectorizer.get_vocabulary()
len(words_in_vocab), words_in_vocab[:10]

### Let's take a look at the embedding layer's weights (these are numerical representations of each word)

In [ ]:
model_1.summary()

In [ ]:
# Get the weight matrix of the embedding layer
embed_weights = model_1.get_layer("embedding").get_weights()[0]
print(
    embed_weights.shape
)  # same size as vocab size and embedding_dim (each word is a embedding size vector)

## Recurrent Neural Networks

## Model 2:LSTM (Long short term memory)

In [ ]:
# Create our LSTM Model
from tensorflow.keras import layers

inputs = layers.Input(shape=(1,), dtype="string")
x = text_vectorizer(inputs)
x = embedding(x)
print(x.shape)
# x = layers.LSTM(64, return_sequences=True)(x) # return vector for each word in the Tweet (you can stack RNN cells as long as return_sequences=True)
x = layers.LSTM(64)(x)  # return vector for whole sequence
print(x.shape)
# x = layers.Dense(64, activation="relu")(x) # optional dense layer on top of output of LSTM cell
outputs = layers.Dense(1, activation="sigmoid")(x)
model_2 = tf.keras.Model(inputs, outputs, name="model_2_LSTM")

In [ ]:
# Compile the model
model_2.compile(
    loss="binary_crossentropy",
    optimizer=tf.keras.optimizers.Adam(),
    metrics=["accuracy"],
)

In [ ]:
model_2.summary()

In [ ]:
# Fit the model
model_2_history = model_2.fit(
    train_sentences,
    train_labels,
    epochs=5,
    validation_data=(val_sentences, val_labels),
    callbacks=[create_tensorboard_callback(SAVE_DIR, "LSTM")],
)

In [ ]:
# Make predictions on the validation dataset
model_2_pred_probs = model_2.predict(val_sentences)
model_2_pred_probs.shape, model_2_pred_probs[:10]

In [ ]:
# Round out predictions and reduce to 1-dimensional array
model_2_preds = tf.squeeze(tf.round(model_2_pred_probs))
model_2_preds[:10]

In [ ]:
# Calculate LSTM model results
model_2_results = calculate_results(y_true=val_labels, y_pred=model_2_preds)
model_2_results

In [ ]:
# Compare model 2 to baseline
compare_baseline_to_new_results(baseline_results, model_2_results)

## Model 3:GRU (Gated Recurrent Unit)

In [ ]:
# Build an RNN model using the GRU cell
from tensorflow.keras import layers

inputs = layers.Input(shape=(1,), dtype="string")
x = text_vectorizer(inputs)
x = embedding(x)
x = layers.GRU(64)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model_3 = tf.keras.Model(inputs, outputs, name="Model_3_GRU")

In [ ]:
# Compile the model
model_3.compile(
    loss="binary_crossentropy",
    optimizer=tf.keras.optimizers.Adam(),
    metrics=["accuracy"],
)

In [ ]:
# Get summary of the model
model_3.summary()

### LSTM cells have more trainable parameters than GRU cells

In [ ]:
# Fit the model
model_3_history = model_3.fit(
    train_sentences,
    train_labels,
    epochs=5,
    validation_data=(val_sentences, val_labels),
    callbacks=[create_tensorboard_callback(SAVE_DIR, "GRU")],
)

In [ ]:
# Make predictions on the validation data
model_3_pred_probs = model_3.predict(val_sentences)
model_3_pred_probs.shape, model_3_pred_probs[:10]

In [ ]:
# Convert the prediction probabilities to prediction classes
model_3_preds = tf.squeeze(tf.round(model_3_pred_probs))
model_3_preds[:10]

In [ ]:
# Calculate model_3 results
model_3_results = calculate_results(y_true=val_labels, y_pred=model_3_preds)

model_3_results

In [ ]:
# Comapare to baseline
compare_baseline_to_new_results(baseline_results, model_3_results)

## Model 4:Bidirectional RNN model

In [ ]:
# Build bidirectional RNN with tensorflow
from tensorflow.keras import layers

inputs = layers.Input(shape=(1,), dtype="string")
x = text_vectorizer(inputs)
x = embedding(x)
x = layers.Bidirectional(layers.LSTM(64))(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model_4 = tf.keras.Model(inputs, outputs, name="model_4_Bidirectional_RNN")

In [ ]:
# Compile the model
model_4.compile(
    loss="binary_crossentropy",
    optimizer=tf.keras.optimizers.Adam(),
    metrics=["accuracy"],
)

In [ ]:
# Get model summary
model_4.summary()

In [ ]:
# Fit the model
model_4_history = model_4.fit(
    train_sentences,
    train_labels,
    epochs=5,
    validation_data=(val_sentences, val_labels),
    callbacks=[create_tensorboard_callback(SAVE_DIR, "Bidirectional_RNN")],
)

In [ ]:
# Make predictions with bidirectional RNN on validation data
model_4_pred_probs = model_4.predict(val_sentences)

In [ ]:
model_4_pred_probs[:10]

In [ ]:
# Convert the prediction probabilities to labels
model_4_preds = tf.squeeze(tf.round(model_4_pred_probs))
model_4_preds[:10]

In [ ]:
# Calculate bidirectional RNN results
model_4_results = calculate_results(y_true=val_labels, y_pred=model_4_preds)
model_4_results

In [ ]:
# Check performance against baseline model
compare_baseline_to_new_results(baseline_results, model_4_results)

## Using CNNs for Text

## Model 5:Conv1D (Temporal Convolution)

In [ ]:
# Test out the embedding, 1D convolution and max pooling
embedding_test = embedding(
    text_vectorizer(["this is a test sentence"])
)  # turn target sentence into embedding
conv_1d = layers.Conv1D(
    filters=32, kernel_size=5, activation="relu"
)  # convolve over target sequence 5 words at a time
conv_1d_output = conv_1d(embedding_test)  # pass embedding through 1D convolutional layer
max_pool = layers.GlobalAveragePooling1D()
max_pool_output = max_pool(conv_1d_output)  # get the most important features
embedding_test.shape, conv_1d_output.shape, max_pool_output.shape

In [ ]:
# See the outputs of each layer
embedding_test[:1], conv_1d_output[:1], max_pool_output[:1]

In [ ]:
# Create 1D convolutional layer to model sequences
from tensorflow.keras import layers

inputs = layers.Input(shape=(1,), dtype="string")
x = text_vectorizer(inputs)
x = embedding(x)
x = layers.Conv1D(filters=32, kernel_size=5, activation="relu")(x)
x = layers.GlobalMaxPool1D()(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model_5 = tf.keras.Model(inputs, outputs, name="model_5_Conv1D")

# Compile the model
model_5.compile(
    loss="binary_crossentropy",
    optimizer=tf.keras.optimizers.Adam(),
    metrics=["accuracy"],
)

# Summary of the model
model_5.summary()

In [ ]:
# Fit the model
model_5_history = model_5.fit(
    train_sentences,
    train_labels,
    epochs=5,
    validation_data=(val_sentences, val_labels),
    callbacks=[create_tensorboard_callback(SAVE_DIR, "Conv1D")],
)

In [ ]:
# Make predictions with model 5
model_5_pred_probs = model_5.predict(val_sentences)
model_5_pred_probs[:10]

In [ ]:
# Convert model_5 prediction probabilities to labels
model_5_preds = tf.squeeze(tf.round(model_5_pred_probs))
model_5_preds[:10]

In [ ]:
# Calculate model_5 evaluation metrics
model_5_results = calculate_results(y_true=val_labels, y_pred=model_5_preds)

model_5_results

In [ ]:
# Compare model 5 to baseline
compare_baseline_to_new_results(baseline_results, model_5_results)

## Model 6:Tensorsflow hub pretrained sentence encoder (encoder-decoder model)


In [ ]:
# Example of pretrained embedding with universal sentence encoder
import tensorflow_hub as hub

embed = hub.load(
    "https://tfhub.dev/google/universal-sentence-encoder/4"
)  # load universal sentence encoder
embed_samples = embed(
    [
        sample_sentence,
        "when you call universal sentence encoder on a sentence, it turns it into numbers",
    ]
)

print(embed_samples[0][:50])

In [ ]:
# Each sentence has been encoded to a 512 dimension vector
embed_samples.shape

In [ ]:
# We can use this encoding layer in place of our text_vectorizer and embedding layer
sentence_encoder_layer = hub.KerasLayer(
    "https://tfhub.dev/google/universal-sentence-encoder/4",
    input_shape=[],  # shape of inputs coming to our model
    dtype=tf.string,  # data type of inputs coming to the USE layer
    trainable=False,  # keep the pretrained weights (Feature Extraction)
    name="USE",
)

In [ ]:
# Create model using sequential API
from experiment_nlp_with_disaster_tweets import build_model_6_cell_113

model_6 = build_model_6_cell_113(sentence_encoder_layer)

# Compile the model
model_6.compile(
    loss="binary_crossentropy",
    optimizer=tf.keras.optimizers.Adam(),
    metrics=["accuracy"],
)

# Summary of the model
model_6.summary()

In [ ]:
# Train a classifier on top of pretrained embeddings
model_6_history = model_6.fit(
    train_sentences,
    train_labels,
    epochs=5,
    validation_data=(val_sentences, val_labels),
    callbacks=[create_tensorboard_callback(SAVE_DIR, "tf_hub_sentence_encoder")],
)

In [ ]:
# Make predictions on the model
model_6_pred_probs = model_6.predict(val_sentences)
model_6_pred_probs[:10]

In [ ]:
# Convert prediction probabilities to labels
model_6_preds = tf.squeeze(tf.round(model_6_pred_probs))
model_6_preds[:10]

In [ ]:
# Calculate model 6 performance metrics
model_6_results = calculate_results(val_labels, model_6_preds)
model_6_results

In [ ]:
# Compare to baseline model
compare_baseline_to_new_results(baseline_results, model_6_results)

## Model 7: Tensorflow Hub pretrained sentence encoder on 10% of training data

In [ ]:
# Split the training data into 90/10 ratios
(
    train_sentences_90_percent,
    train_sentences_10_percent,
    train_labels_90_percent,
    train_labels_10_percent,
) = train_test_split(np.array(train_sentences), train_labels, test_size=0.1, random_state=42)

In [ ]:
# Check the length of targets in out subset of data
pd.Series(train_labels_10_percent).value_counts()

In [ ]:
# Clone model 6 but reset weights
model_7 = tf.keras.models.clone_model(model_6)

# Compile the model
model_7.compile(
    loss="binary_crossentropy",
    optimizer=tf.keras.optimizers.Adam(),
    metrics=["accuracy"],
)

# Get summary
model_7.summary()

In [ ]:
# Fit the model on 10% training data
model_7_history = model_7.fit(
    x=train_sentences_10_percent,
    y=train_labels_10_percent,
    epochs=5,
    validation_data=(val_sentences, val_labels),
    callbacks=[create_tensorboard_callback(SAVE_DIR, "10_percent_tf_hub_sentence encoder")],
)

In [ ]:
# Make predictions with the model trained on 10% of the data
model_7_pred_probs = model_7.predict(val_sentences)
model_7_pred_probs[:10]

In [ ]:
# Convert prediction probabilities to labels
model_7_preds = tf.squeeze(tf.round(model_7_pred_probs))
model_7_preds[:10]

In [ ]:
# Calculate model results
model_7_results = calculate_results(val_labels, model_7_preds)
model_7_results

In [ ]:
# Compare to baseline
compare_baseline_to_new_results(baseline_results, model_7_results)

# Comparing the performance of each of our models

In [ ]:
# Combine model results into a DataFrame
all_model_results = pd.DataFrame(
    {
        "baseline": baseline_results,
        "simple_dense": model_1_results,
        "lstm": model_2_results,
        "gru": model_3_results,
        "bidirectional": model_4_results,
        "conv1d": model_5_results,
        "tf_hub_sentence_encoder": model_6_results,
        "tf_hub_10_percent_data": model_7_results,
    }
)

all_model_results = all_model_results.transpose()
all_model_results

In [ ]:
# Reduce the accuracy to same scale as other metrics
all_model_results["accuracy"] = all_model_results["accuracy"] / 100

In [ ]:
# Plot and compare all of the models
all_model_results.plot(kind="bar", figsize=(10, 7)).legend(bbox_to_anchor=(1.0, 1.0));

In [ ]:
# Sort model results by f1 score
all_model_results.sort_values("f1", ascending=False)["f1"].plot(kind="bar", figsize=(10, 7))

# Combining our Models

Many production systems use an ensemble (multiple different models combined) of models to make a prediction. The idea behind model stacking is that if several uncorrelated models agree on a prediction, then the prediction must be more robust than a prediction made by singular model. Uncorrelated means different kinds of models like models that use different algorithms. There are a few ways we can combine our models:
* Averaging - Take the output prediction probabilities of each model for each sample, combine them and average them.
* Majority Vote - Make class predictions with each of your models on all samples, the predicted class is the one in majority.
* Model Stacking - Take the outputs of each of your chosen models and use them as inputs to another model

In [ ]:
# Get mean pred probs for 3 models
baseline_pred_probs = np.max(
    model_0.predict_proba(val_sentences), axis=1
)  # get prediction probabilities from baseline model
combined_pred_probs = (
    baseline_pred_probs + tf.squeeze(model_2_pred_probs, axis=1) + tf.squeeze(model_6_pred_probs)
)
combined_preds = tf.round(combined_pred_probs / 3)
combined_preds[:20]

In [ ]:
# Calculate results from averaging the prediction probabilities
ensemble_results = calculate_results(val_labels, combined_preds)
ensemble_results

In [ ]:
# Add our combined model's results to the results DataFrame
all_model_results.loc["ensemble_results"] = ensemble_results

In [ ]:
# Convert the accuracy to the same scale as the rest of the results
all_model_results.loc["ensemble_results"]["accuracy"] = (
    all_model_results.loc["ensemble_results"]["accuracy"] / 100
)

In [ ]:
all_model_results

# Saving and Loading a trained model

In [ ]:
# Save TF Hub Sentence Encoder model to HD5 format
model_6.save("model_6.h5")

In [ ]:
# Load model with custom Hub Layer (required with HDF5 formay)
loaded_model_6 = tf.keras.models.load_model(
    "model_6.h5", custom_objects={"KerasLayer": hub.KerasLayer}
)

In [ ]:
# How does our loaded model perform
loaded_model_6.evaluate(val_sentences, val_labels)

In [ ]:
# Save TF Hub Sentence Encoder model to SavedModel format (default)
model_6.save("model_6_native.keras")

In [ ]:
# Load TF Hub Sentence Encoder SavedModel
loaded_model_6_SavedModel = tf.keras.models.load_model(
    "model_6_native.keras", custom_objects={"KerasLayer": hub.KerasLayer}
)

In [ ]:
# Evaluate loaded savedmodel format
loaded_model_6_SavedModel.evaluate(val_sentences, val_labels)

# Finding the most wrong examples in our data

In [ ]:
# Create dataframe with validation sentences and best perfomring model preditctions
val_df = pd.DataFrame(
    {
        "text": val_sentences,
        "target": val_labels,
        "pred": model_6_preds,
        "pred_prob": tf.squeeze(model_6_pred_probs),
    }
)

val_df.head(10)

In [ ]:
# Find the wrong predictions and sort by prediction probabilities
most_wrong = val_df[val_df["target"] != val_df["pred"]].sort_values("pred_prob", ascending=False)
most_wrong[:10]

In [ ]:
# Check the false positives (model predicted 1 when should've been 0)
for row in most_wrong[
    :10
].itertuples():  # loop through the top 10 rows (change the index to view different rows)
    _, text, target, pred, prob = row
    print(f"Target: {target}, Pred: {int(pred)}, Prob: {prob}")
    print(f"Text:\n{text}\n")
    print("----\n")

In [ ]:
# Check the most wrong false negatives (model predicted 0 when should've predicted 1)
for row in most_wrong[
    -10:
].itertuples():  # the last rows in the dataframe contain the false negatives
    _, text, target, pred, prob = row
    print(f"Target: {target}, Pred: {int(pred)}, Prob: {prob}")
    print(f"Text:\n{text}\n")
    print("----\n")